# 02. PINN CPU Validation

**v6 Section 17.10.1** | Phase C Development Notebook #2

## Purpose
작은 모델(hidden_dim=64, 3layers)로 **구조 검증** (집 CPU)

## Success Criteria (v6)
- Loss 발산 없이 감소
- Slit/BM 분화 조짐
- z 내부 약간의 fringe (Stage 2 이후)

## Reference
- v6 Section 5: PINN architecture (8D, SIREN+Fourier)
- v6 Section 6: Loss (L_H, L_phase, L_BC)
- v6 Section 7: Curriculum 3-Stage
- v6 Section 8: Hierarchical collocation sampling

---
## Cell 1: Import & Setup

In [ ]:
import sys
from pathlib import Path

# Robust project root detection
def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Cannot find project root")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from backend.core.pinn_model import PurePINN
from backend.training.loss_functions import (
    helmholtz_loss, phase_loss, bm_boundary_loss, ASMIncidentLUT,
)
from backend.training.collocation_sampler import hierarchical_collocation
from backend.training.curriculum import CurriculumConfig, get_loss_weights, get_stage_name

DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

---
## Cell 2: Config (Small CPU)

v6 Section 15.5: 작게 시작, 구조 검증

In [ ]:
# Small model for CPU validation
CONFIG = {
    'hidden_dim': 64,
    'num_layers': 3,
    'num_freqs': 24,
    'omega_0': 30.0,
    'n_colloc': 2000,
    'n_bc': 200,
    'n_phase': 200,
    'epochs': 300,
    'lr': 1e-3,
}

curriculum = CurriculumConfig(
    total_epochs=CONFIG['epochs'],
    stage1_frac=0.20,   # 0~60: boundary only
    stage2_frac=0.40,   # 60~180: PDE ramp
    # 180~300: full
)

print(f"Stage 1 (Boundary): epoch 0 ~ {curriculum.stage1_end}")
print(f"Stage 2 (PDE ramp): epoch {curriculum.stage1_end} ~ {curriculum.stage2_end}")
print(f"Stage 3 (Full):     epoch {curriculum.stage2_end} ~ {curriculum.total_epochs}")

---
## Cell 3: Model & Optimizer

In [ ]:
model = PurePINN(
    hidden_dim=CONFIG['hidden_dim'],
    num_layers=CONFIG['num_layers'],
    num_freqs=CONFIG['num_freqs'],
    omega_0=CONFIG['omega_0'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=1e-5
)

# Load ASM LUT for L_phase
asm_lut = ASMIncidentLUT(str(PROJECT_ROOT / 'data' / 'asm_luts' / 'incident_z40.npz'))
print('ASM LUT loaded.')

---
## Cell 4: Training Loop (with live visualization)

v6 Section 7: Curriculum 3-Stage
- Stage 1: L_phase + L_BC (boundary learning)
- Stage 2: + L_H ramp (PDE activation)
- Stage 3: Full loss

In [ ]:
from IPython.display import clear_output

history = {'L_total': [], 'L_H': [], 'L_phase': [], 'L_BC': [], 'stage': []}

for epoch in range(CONFIG['epochs']):
    model.train()
    weights = get_loss_weights(epoch, curriculum)
    
    # Collocation points (fresh each epoch)
    coords = hierarchical_collocation(CONFIG['n_colloc'], DEVICE)
    
    # Compute losses
    L_H = helmholtz_loss(model, coords) if weights['lambda_H'] > 0 else torch.tensor(0.0)
    L_ph = phase_loss(model, asm_lut, CONFIG['n_phase'], DEVICE)
    L_bc = bm_boundary_loss(model, CONFIG['n_bc'], DEVICE)
    
    L_total = (
        weights['lambda_H'] * L_H
        + weights['lambda_phase'] * L_ph
        + weights['lambda_BC'] * L_bc
    )
    
    optimizer.zero_grad()
    L_total.backward()
    optimizer.step()
    scheduler.step()
    
    # Record
    history['L_total'].append(L_total.item())
    history['L_H'].append(L_H.item())
    history['L_phase'].append(L_ph.item())
    history['L_BC'].append(L_bc.item())
    history['stage'].append(get_stage_name(epoch, curriculum))
    
    # Live visualization every 50 epochs
    if (epoch + 1) % 50 == 0 or epoch == 0:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        ep_range = range(len(history['L_total']))
        
        # Loss curves
        ax = axes[0]
        ax.semilogy(ep_range, history['L_total'], 'k-', label='Total', linewidth=1.5)
        ax.semilogy(ep_range, history['L_H'], 'b-', label='L_H', alpha=0.7)
        ax.semilogy(ep_range, history['L_phase'], 'r-', label='L_phase', alpha=0.7)
        ax.semilogy(ep_range, history['L_BC'], 'g-', label='L_BC', alpha=0.7)
        ax.axvline(x=curriculum.stage1_end, color='orange', linestyle='--', alpha=0.5)
        ax.axvline(x=curriculum.stage2_end, color='purple', linestyle='--', alpha=0.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss (log)')
        ax.set_title(f'Epoch {epoch+1} | {get_stage_name(epoch, curriculum)}')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        # z=40 cross-section (BM2 boundary check)
        ax = axes[1]
        model.eval()
        with torch.no_grad():
            x_vis = torch.linspace(0, 504, 500)
            z_vis = torch.full((500,), 40.0)
            d1_vis = torch.zeros(500)
            d2_vis = torch.zeros(500)
            w1_vis = torch.full((500,), 10.0)
            w2_vis = torch.full((500,), 10.0)
            sin_vis = torch.zeros(500)
            cos_vis = torch.ones(500)
            coords_vis = torch.stack([x_vis, z_vis, d1_vis, d2_vis, w1_vis, w2_vis, sin_vis, cos_vis], dim=1)
            U_vis = model(coords_vis)
            amp_vis = torch.sqrt(U_vis[:,0]**2 + U_vis[:,1]**2)
        
        ax.plot(x_vis.numpy(), amp_vis.numpy(), 'b-', linewidth=0.8)
        # Mark slit centers
        for i in range(7):
            cx = i * 72 + 36
            ax.axvspan(cx - 5, cx + 5, alpha=0.1, color='green', label='slit' if i == 0 else '')
        ax.set_xlabel('x (μm)')
        ax.set_ylabel('|U|')
        ax.set_title('z=40 (BM2): |U|(x), θ=0°')
        ax.grid(True, alpha=0.3)
        
        # z=10 cross-section (interior fringe check)
        ax = axes[2]
        with torch.no_grad():
            z_vis_10 = torch.full((500,), 10.0)
            coords_10 = torch.stack([x_vis, z_vis_10, d1_vis, d2_vis, w1_vis, w2_vis, sin_vis, cos_vis], dim=1)
            U_10 = model(coords_10)
            amp_10 = torch.sqrt(U_10[:,0]**2 + U_10[:,1]**2)
        
        ax.plot(x_vis.numpy(), amp_10.numpy(), 'r-', linewidth=0.8)
        ax.set_xlabel('x (μm)')
        ax.set_ylabel('|U|')
        ax.set_title('z=10 (Encap interior): |U|(x)')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f'Epoch {epoch+1}/{CONFIG["epochs"]} | '
              f'Total={L_total.item():.4f} H={L_H.item():.4f} '
              f'Phase={L_ph.item():.4f} BC={L_bc.item():.4f} '
              f'LR={scheduler.get_last_lr()[0]:.2e}')

print('\nTraining complete.')

---
## Cell 5: Post-Training Visualization

v6 Section 17.10.1:
- z=10 단면 |U|(x)
- z=40 단면 |U|(x) vs ASM
- z=20, z=40 BM 영역 |U| 값

In [ ]:
model.eval()

# Design variables for visualization: delta=0, w=10, theta=0
N_VIS = 1000
x_vis = torch.linspace(0, 504, N_VIS)
d1_v = torch.zeros(N_VIS)
d2_v = torch.zeros(N_VIS)
w1_v = torch.full((N_VIS,), 10.0)
w2_v = torch.full((N_VIS,), 10.0)
sin_v = torch.zeros(N_VIS)  # theta=0
cos_v = torch.ones(N_VIS)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) z=40: PINN vs ASM at BM2
ax = axes[0, 0]
with torch.no_grad():
    z_40 = torch.full((N_VIS,), 40.0)
    coords_40 = torch.stack([x_vis, z_40, d1_v, d2_v, w1_v, w2_v, sin_v, cos_v], dim=1)
    U_40 = model(coords_40)
    amp_40 = torch.sqrt(U_40[:,0]**2 + U_40[:,1]**2).numpy()

U_asm_re, U_asm_im = asm_lut.lookup(x_vis, sin_v)
amp_asm = torch.sqrt(U_asm_re**2 + U_asm_im**2).numpy()

ax.plot(x_vis.numpy(), amp_40, 'b-', linewidth=0.8, label='PINN')
ax.plot(x_vis.numpy(), amp_asm, 'r--', linewidth=0.8, label='ASM target')
ax.set_title('z=40 (BM2): PINN vs ASM, θ=0°')
ax.set_xlabel('x (μm)')
ax.set_ylabel('|U|')
ax.legend()
ax.grid(True, alpha=0.3)

# (b) z=20: BM1 boundary check
ax = axes[0, 1]
with torch.no_grad():
    z_20 = torch.full((N_VIS,), 20.0)
    coords_20 = torch.stack([x_vis, z_20, d1_v, d2_v, w1_v, w2_v, sin_v, cos_v], dim=1)
    U_20 = model(coords_20)
    amp_20 = torch.sqrt(U_20[:,0]**2 + U_20[:,1]**2).numpy()

ax.plot(x_vis.numpy(), amp_20, 'b-', linewidth=0.8)
for i in range(7):
    cx = i * 72 + 36
    ax.axvspan(cx - 5, cx + 5, alpha=0.15, color='green')
ax.set_title('z=20 (BM1): |U|(x), θ=0°')
ax.set_xlabel('x (μm)')
ax.set_ylabel('|U|')
ax.grid(True, alpha=0.3)

# (c) z=10: Interior fringe check
ax = axes[1, 0]
with torch.no_grad():
    z_10 = torch.full((N_VIS,), 10.0)
    coords_10 = torch.stack([x_vis, z_10, d1_v, d2_v, w1_v, w2_v, sin_v, cos_v], dim=1)
    U_10 = model(coords_10)
    amp_10 = torch.sqrt(U_10[:,0]**2 + U_10[:,1]**2).numpy()

ax.plot(x_vis.numpy(), amp_10, 'r-', linewidth=0.8)
ax.set_title(f'z=10 (Encap): |U|(x), std={np.std(amp_10):.4f}')
ax.set_xlabel('x (μm)')
ax.set_ylabel('|U|')
ax.grid(True, alpha=0.3)

# (d) 2D field map (x vs z)
ax = axes[1, 1]
z_range = torch.linspace(0, 40, 100)
x_range = torch.linspace(0, 504, 200)
Z, X = torch.meshgrid(z_range, x_range, indexing='ij')
N_map = Z.numel()

with torch.no_grad():
    coords_map = torch.stack([
        X.flatten(),
        Z.flatten(),
        torch.zeros(N_map),
        torch.zeros(N_map),
        torch.full((N_map,), 10.0),
        torch.full((N_map,), 10.0),
        torch.zeros(N_map),
        torch.ones(N_map),
    ], dim=1)
    U_map = model(coords_map)
    amp_map = torch.sqrt(U_map[:,0]**2 + U_map[:,1]**2).reshape(100, 200).numpy()

im = ax.imshow(amp_map, aspect='auto', origin='lower',
               extent=[0, 504, 0, 40], cmap='hot')
ax.axhline(y=20, color='cyan', linestyle='--', alpha=0.5, label='BM1')
ax.axhline(y=40, color='cyan', linestyle='-', alpha=0.5, label='BM2')
ax.set_xlabel('x (μm)')
ax.set_ylabel('z (μm)')
ax.set_title('|U|(x, z) 2D field map, θ=0°')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Cell 6: Red Flag Check

v6 Section 18.5 / 10개 계명 #10:
- 평면파 수렴 체크 (z 내부 uniform → FAIL)
- BM 영역 |U| 체크 (> 0.05 → FAIL)
- 설계변수 반응 체크

In [ ]:
print('='*60)
print('RED FLAG CHECK (v6 Section 18.5)')
print('='*60)

model.eval()

# Check 1: Plane wave convergence (interior should NOT be uniform)
with torch.no_grad():
    z_interior = torch.full((500,), 10.0)  # Encap interior
    x_check = torch.linspace(0, 504, 500)
    coords_check = torch.stack([
        x_check, z_interior,
        torch.zeros(500), torch.zeros(500),
        torch.full((500,), 10.0), torch.full((500,), 10.0),
        torch.zeros(500), torch.ones(500),
    ], dim=1)
    U_check = model(coords_check)
    amp_check = torch.sqrt(U_check[:,0]**2 + U_check[:,1]**2)

interior_std = amp_check.std().item()
interior_mean = amp_check.mean().item()
is_uniform = interior_std / max(interior_mean, 1e-8) < 0.05  # < 5% CoV = uniform

print(f'\n[Check 1] Interior uniformity (z=10):')
print(f'  |U| mean={interior_mean:.4f}, std={interior_std:.4f}')
print(f'  CoV={interior_std/max(interior_mean,1e-8)*100:.2f}%')
if is_uniform:
    print('  WARNING: Interior is uniform - possible plane wave convergence')
    print('  (Expected for 300 epoch CPU test - full GPU training should show fringes)')
else:
    print('  OK: Interior shows variation')

# Check 2: BM region |U| (should approach 0)
from backend.physics.boundary_conditions import compute_is_bm

with torch.no_grad():
    # BM1 at z=20
    is_bm1 = compute_is_bm(x_check, torch.zeros(500), torch.full((500,), 10.0))
    amp_bm1 = amp_20[is_bm1.numpy()] if is_bm1.sum() > 0 else np.array([0])
    
    # BM2 at z=40
    is_bm2 = compute_is_bm(x_check, torch.zeros(500), torch.full((500,), 10.0))
    amp_bm2_vals = amp_40[is_bm2.numpy()] if is_bm2.sum() > 0 else np.array([0])

print(f'\n[Check 2] BM region |U| (target: < 0.05):')
print(f'  BM1 (z=20): mean={np.mean(amp_bm1):.4f}, max={np.max(amp_bm1):.4f}')
print(f'  BM2 (z=40): mean={np.mean(amp_bm2_vals):.4f}, max={np.max(amp_bm2_vals):.4f}')

# Check 3: Design variable sensitivity
print(f'\n[Check 3] Design variable sensitivity:')
with torch.no_grad():
    N_s = 200
    x_s = torch.full((N_s,), 252.0)  # center
    z_s = torch.full((N_s,), 0.0)     # OPD
    sin_s = torch.zeros(N_s)
    cos_s = torch.ones(N_s)
    
    # Vary w1 from 5 to 20
    w1_sweep = torch.linspace(5, 20, N_s)
    coords_sweep = torch.stack([
        x_s, z_s, torch.zeros(N_s), torch.zeros(N_s),
        w1_sweep, torch.full((N_s,), 10.0),
        sin_s, cos_s,
    ], dim=1)
    U_sweep = model(coords_sweep)
    amp_sweep = torch.sqrt(U_sweep[:,0]**2 + U_sweep[:,1]**2)
    
sweep_range = amp_sweep.max().item() - amp_sweep.min().item()
print(f'  w1 sweep [5, 20]: |U| range = {sweep_range:.4f}')
print(f'  {"OK: Shows sensitivity" if sweep_range > 0.01 else "WARNING: Low sensitivity"}')

# Summary
print(f'\n{"="*60}')
print('Note: This is a small CPU validation (64-dim, 300 epochs).')
print('Full GPU training (128-dim, 50K epochs) expected for real convergence.')
print(f'{"="*60}')

---
## Cell 7: Conclusion & Next Steps

### Expected Results (CPU small model)
- Loss should decrease without divergence
- BM regions should show decreasing |U| (even if not yet < 0.05)
- Interior may still look uniform at 300 epochs (need more training)

### Next Steps
1. If structure is validated → proceed to GPU training with `scripts/train_phase_c.py`
2. If loss diverges → adjust learning rate or architecture
3. If BM not learning → increase L_BC weight or sampling density

→ `03_pinn_training_monitor.ipynb`: Monitor GPU training progress